# 2.0. Product metadata

The `product` STAC Collection provides a general metadata description of all project outputs which will be discovered on the Open Science Catalogue (OSC). Most of these metadata fields should already be available and can be extracted from your data or documentation.

This notebook shows how to create an OSC product entry using the current `earthcode` API, save it in a local OSC catalog clone, add an item to the product, and validate the full catalog.

> You can **attach one or more products to a single project**! So if you have more than one, you have run the cell [2-4] for each product to add !


> See **example product metadata** directly at open science catalogue metadata repository on GitHub to compare the list of required parameters and their format: [See example product: WAPOSAL Dataset](https://github.com/ESA-EarthCODE/open-science-catalog-metadata/blob/main/products/waposal-waves/collection.json)


**LICENSE:** In this step you are required to select one of the available licenses for **each of your product**. <br> Please have a look at available list of license and pick the one that defines your datasets: [osc-licence schemas](https://github.com/ESA-EarthCODE/open-science-catalog-validation/blob/main/schemas/license.json). <br>
>*If you have a product with non-defined license, we **cannot proceed with publishing the datasets**. Please use the list of [licenses by SPDX](https://spdx.org/licenses/) and select the most appropriate one.* <br>
> Visit EarthCODE Best Practices to learn more about [**Open Data & Licensing**](https://esa-earthcode.github.io/documentation/Community%20and%20Best%20Practices/FAIR%20and%20Open%20Science%20Best%20Practices/Data#open-data-licensing)<br>

**PRODUCT EO-MISSIONS:** In this step you are required to select one or more EO Missions that you have used to generate your product.
Please have a look at the **defined list of EO missions** available in the OSC under: (https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/eo-missions) and searchable under: https://opensciencedata.esa.int/eo-missions/catalog <br> 
>*If you have a product which uses or complements in-situ data collections or comes as a results of numerical models please select: ["in-situ-observations"] or ["numerical-models"]*<br>

**PRODUCT VARIABLES:** In this step you are required to select one or more variables that your product describes.
Please have a look at the **defined list of geophysical variables** available in the OSC under: (https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/variables). You can also explore the list of variables under: https://opensciencedata.esa.int/variables/catalog

> Variables are defined in OSC as geophysical, climate and environmental variables selected from [WMO OSCAR Database](https://space.oscar.wmo.int/variables), complemented by the [GCMD Keywords Database](https://gcmd.earthdata.nasa.gov/KeywordViewer/scheme/Earth%20Science?gtm_scheme=Earth%20Science)

*NOTE: You can use the [EarthCODE search](examples/earthcode_data_discovery.ipynb) functionality to find relevant variables and eo-missions to your data.*

**PRODUCT PARAMETER:** 
Please provide a parameter linked to the product, in allignment with the **CF convention** standard: See full list under: [https://cfconventions.org/](https://cfconventions.org/)

**PRODUCT DOI:**
Since few weeks EarthCODE offers DOI assignment to products/datasets published on Open Science Data Catalogue. The process is still manual and is handled by ESA TEllUs service, and is handled by the EarthCODE Data Stewardship team on behalf of the Project PI. 
> If you would like to assign a DOI to your data, please contact the EarthCODE Team, who will support you in this process: [earth-code@esa.int](mailto:earth-code@esa.int)!


In [1]:
import os
from datetime import datetime
from pathlib import Path

from earthcode.metadata_input_definitions import ProductCollectionMetadata
from earthcode.static import create_product_collection
from earthcode.git_add import save_product_collection_to_catalog

from earthcode.search import search
import rasterio
from rasterio.warp import transform_bounds

# Submit individual maps

In [1]:
# Local OSC clone root path (assumed one folder above repository root)
catalog_root = str(Path("../../../open-science-catalog-metadata").resolve())
file_root = r"C:\Projects\BLS-BIOMASS\BiomassMaps"
suffix_cog = "FCT-ISA-NMBU_Map.tif"
locations = [
    ["Mafra", "Portugal"],
    ["Monsanto", "Portugal"],
    ["Pombal", "Portugal"],
    ["SerraLousa", "Portugal"],
    ["SintraCascais", "Portugal"],
    ["VPA", "Portugal"],
    ["Extremadura", "Spain"],
    ["HuescaZaragoza", "Spain"],
    ["Leon", "Spain"],
    ["Teruel", "Spain"],

]
biomassPools = [
    ["AGBD", "AboveGround Biomass Density Map", "aboveground", "10.5281/zenodo.24346310"],
    ["BGBD", "BelowGround Biomass Density Map", "belowground", "10.5281/zenodo.24347014"],
    ["LBD", "Litter Biomass Density Map", "litter", "10.5281/zenodo.24347321"],
    ["SBD", "Shrub Biomass Density Map", "shrub", "10.5281/zenodo.24347392"],
]

metrics = {
    "CV": "Coefficient of Variation",
    "IQR": "Interquartile Range",
    "KURTOSIS": "Kurtosis",
    "MEAN": "Mean",
    "MEDIAN": "Median",
    "RELATIVESTANDARDERROR": "Relative Standard Error",
    "SKEWNESS": "Skewness",
    "STANDARDERROR": "Standard Error",

}

suffix = "FCT-ISA-NMBU_Map.tif"

for biomassPool, biomassPoolDesc, biomassPoolProd_id, biomassPoolDOI in biomassPools:

    # A custom id of the product (must be different from project!), it can be related to the title, i.e. - 4datlantic-ohc-dataset. Use dash "-" symbol to separare words in the id"
    product_id = "biomass-" + biomassPoolProd_id + "-iberian-peninsula-bls-biomass"
    product_title = biomassPoolDesc + " and uncertainty maps for several areas of Portugal and Spain"
    product_description = "This product provides the " + biomassPoolProd_id + " biomass for forest areas in several areas of Portugal and Spain, together with their corresponding uncertainty assessments. The areas in Portugal are Mafra, Monsanto, Pombal, Serra Lousã, Sintra-Cascais and Vila Pouca de Aguiar; and in Spain are Extremadura, Huesca-Zaragoza, Leon and Teruel. For each area the dataset includes individual raster maps in Cloud Optimized GeoTIFF (COG) format, with 25m resolution, for all extracted parameters, as well as a Parquet vector dataset containing the same parameters represented as a point cloud. The following statistical parameters are included: Coefficient of Variation, Interquartile Range, Kurtosis, Mean, Median, Relative Standard Error, Skewness, and Standard Error."

    # Product status: pick from - ongoing or completed
    product_status = "completed"

    # Define the product license. i.e. CC-BY-4.0. See the note in the markdown cell above to consult full list of available licenses.
    # If you have a license agreement that is not in the list, you can put 'other' in the product licenses and provide a link to the text directly
    # if your license is in the list, leave the license link as None.
    product_license = "CC-BY-4.0"
    license_link = None

    # Define at most five keywords for the product. You can use any short text, that allow users to discover your product.
    product_keywords = ["biomass", "carbon", "forest", "shrub"]

    # Define the temporal extent of PRODUCT/ DATASET
    product_start_year = 2021
    product_start_month = 1
    product_start_day = 1
    product_end_year = 2021
    product_end_month = 12
    product_end_day = 31

    # Define the semantic region covered by this product, i.e. Belgium, Global etc.
    product_region = "Iberian Peninsula"

    # Define product themes i.e. land. Pick one or more from:
    # - atmosphere, cryosphere, land, magnetosphere-ionosphere, oceans, solid-earth.
    # See the list here: https://github.com/ESA-    EarthCODE/open-science-catalog-metadata/blob/main/themes/catalog.json
    product_themes = ["land"]

    # Define the eo-mission(s) used to generate the product. i.e. - "sentinel-2"
    # Pick one or more from - https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/eo-missions
    product_missions = ["Sentinel-1", "Sentinel-2", "ALOS-2"]

    # Define variables describing at best your Product/ dataset:
    # Pick one or more from from https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/variables
    product_variables = ["biomass"]

    # search("description",type="variable")

    # Provide DOI number assigned to your product. If your product does not have one, type: None
    product_doi = biomassPoolDOI

    # Define the related project id and title
    # These must match the new or an already existing project in the catalog! Alteratively correct links cannot be produced!
    project_id = "bls-biomass"
    project_title = "Below, Litter & Shrub BIOMASS density mapping combining optical, LiDAR and SAR Earth Observation data"


    product_w = []
    product_s = []
    product_e = []
    product_n = []
    for location, locationCountry in locations:

        product_parameters = []
        for metric, metricDesc in metrics.items():
            # Define spatial extent of PRODUCT/DATASET in epsg:4326. If the dataset covers discontinuous regions,
            # add the bounding box boundaries for each
            filename = location + "_" + biomassPool + "_" + metric + "_"
            with rasterio.open(os.path.join(file_root, filename + suffix_cog)) as src:
                bounds = transform_bounds(
                    src.crs,
                    "EPSG:4326",
                    *src.bounds
                )

            # Define the parameters describing your product in standardised CF convention format: i.e. "leaf_area_index".
            product_parameters.append(
                biomassPoolDesc.lower().replace(" ", "_") + "_" + metricDesc.lower().replace(" ", "_"))

        product_w.append(bounds[0])
        product_s.append(bounds[1])
        product_e.append(bounds[2])
        product_n.append(bounds[3])

    # create a product
    product_bbox = [[w, s, e, n] for s, w, n, e in zip(product_s, product_w, product_n, product_e)]

    product_metadata = ProductCollectionMetadata(
        product_id=product_id,
        product_title=product_title,
        product_description=product_description,
        product_bbox=product_bbox,
        product_start_datetime=datetime(product_start_year, product_start_month, product_start_day),
        product_end_datetime=datetime(product_end_year, product_end_month, product_end_day),
        product_license=product_license,
        product_keywords=product_keywords,
        product_status=product_status,
        product_region=product_region,
        product_themes=product_themes,
        product_missions=product_missions,
        product_variables=product_variables,
        project_id=project_id,
        project_title=project_title,
        product_parameters=product_parameters,
        product_doi=product_doi,
        access_link=f'https://opensciencedata.esa.int/products/{product_id}/collection',
        license_link=license_link or None,
    )

    product_collection = create_product_collection(product_metadata)

    # save the product in local fork of the open-science-catalog-metadata repository
    catalog_root = Path(catalog_root)
    save_product_collection_to_catalog(product_collection, catalog_root)


NameError: name 'Path' is not defined

# Submit Geopackages (PARQUET)

In [ ]:
# Local OSC clone root path (assumed one folder above repository root)
catalog_root = str(Path("../../../open-science-catalog-metadata").resolve())

locations = [
    "Mafra",
    "Monsanto",
    "Pombal",
    "SerraLousa",
    "SintraCascais",
    "VPA",
]

biomassPools = {
"AGBD":"AboveGround Biomass Density Map Geopackage",
"BGBD":"BelowGround Biomass Density Map Geopackage",
"LBD":"Litter Biomass Density Map Geopackage",
"SBD":"Shrub Biomass Density Map Geopackage",

}

suffix = "Map.gpkg"

for location in locations:
    for biomassPool,biomassPoolDesc in biomassPools.items():

        # A custom id of the product (must be different from project!), it can be related to the title, i.e. - 4datlantic-ohc-dataset. Use dash "-" symbol to separare words in the id"
        product_id = location + "bls-biomass-" + biomassPool
        product_title = location +" " + biomassPoolDesc
        product_description = biomassPoolDesc + " for region " + location
        # Product status: pick from - ongoing or completed
        product_status = "completed"

        # Define the product license. i.e. CC-BY-4.0. See the note in the markdown cell above to consult full list of available licenses.
        # If you have a license agreement that is not in the list, you can put 'other' in the product licenses and provide a link to the text directly
        # if your license is in the list, leave the license link as None.
        product_license = "CC-BY-4.0"
        license_link = None

        # Define at most five keywords for the product. You can use any short text, that allow users to discover your product.
        product_keywords = ["biomass", "carbon"]

        # Define spatial extent of PRODUCT/DATASET in epsg:4326. If the dataset covers discontinuous regions,
        # add the bounding box boundaries for each


        product_w = [-180.0]
        product_s = [-90.0]
        product_e = [180.0]
        product_n = [90.0]

        # Define the temporal extent of PRODUCT/ DATASET
        product_start_year = 2021
        product_start_month = 1
        product_start_day = 1
        product_end_year = 2021
        product_end_month = 12
        product_end_day = 31

        # Define the semantic region covered by this product, i.e. Belgium, Global etc.
        product_region = "ROI in Portugal"

        # Define product themes i.e. land. Pick one or more from:
        # - atmosphere, cryosphere, land, magnetosphere-ionosphere, oceans, solid-earth.
        # See the list here: https://github.com/ESA-EarthCODE/open-science-catalog-metadata/blob/main/themes/catalog.json
        product_themes = [""]

        # Define the eo-misison(s) used to generate the product. i.e. - "sentinel-2"
        # Pick one or more from - https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/eo-missions
        product_missions = []

        # Define variables describing at best your Product/ dataset:
        # Pick one or more from from https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/variables
        product_variables = []

        # search("description",type="variable")

        # Define the parameters describing your product in standardised CF convention format: i.e. "leaf_area_index".
        product_parameters = []

        # Provide DOI number assigned to your product. If your product does not have one, type: None
        product_doi = None

        # Define the related project id and title
        # These must match the new or an already existing project in the catalog! Alteratively correct links cannot be produced!
        project_id = ""
        project_title = ""

        # create a product
        product_bbox = [[w, s, e, n] for s, w, n, e in zip(product_s, product_w, product_n, product_e)]

        product_metadata = ProductCollectionMetadata(
            product_id=product_id,
            product_title=product_title,
            product_description=product_description,
            product_bbox=product_bbox,
            product_start_datetime=datetime(product_start_year, product_start_month, product_start_day),
            product_end_datetime=datetime(product_end_year, product_end_month, product_end_day),
            product_license=product_license,
            product_keywords=product_keywords,
            product_status=product_status,
            product_region=product_region,
            product_themes=product_themes,
            product_missions=product_missions,
            product_variables=product_variables,
            project_id=project_id,
            project_title=project_title,
            product_parameters=product_parameters,
            product_doi=product_doi,
            access_link=f'https://opensciencedata.esa.int/products/{product_id}/collection',
            license_link=license_link or None,
        )

        product_collection = create_product_collection(product_metadata)

        # save the product in local fork of the open-science-catalog-metadata repository
        catalog_root = Path(catalog_root)
        save_product_collection_to_catalog(product_collection, catalog_root)

### Create Project collection
> ℹ️ **Note**  
> This function **creates a product collection** and automatically generates STAC Collection.json and all required **STAC links**.  
> It connects the project with related projects, themes, and missions, and updates existing entries as needed. <br>
> Run the cell below to automatically create new entry. 


In [ ]:
# create a product
product_bbox = [[w, s, e, n] for s, w, n, e in zip(product_s, product_w, product_n, product_e)]

product_metadata = ProductCollectionMetadata(
    product_id=product_id,
    product_title=product_title,
    product_description=product_description,
    product_bbox=product_bbox,
    product_start_datetime=datetime(product_start_year, product_start_month, product_start_day),
    product_end_datetime=datetime(product_end_year, product_end_month, product_end_day),
    product_license=product_license,
    product_keywords=product_keywords,
    product_status=product_status,
    product_region=product_region,
    product_themes=product_themes,
    product_missions=product_missions,
    product_variables=product_variables,
    project_id=project_id,
    project_title=project_title,
    product_parameters=product_parameters,
    product_doi=product_doi,
    access_link=f'https://opensciencedata.esa.int/products/{product_id}/collection',
    license_link=license_link or None,
)

product_collection = create_product_collection(product_metadata)

### Save the product entry into local fork of the open-science-catalog-metadata repository

In [ ]:
# save the product in local fork of the open-science-catalog-metadata repository 
catalog_root = Path(catalog_root)
save_product_collection_to_catalog(product_collection, catalog_root)

In this case we do not run validation yet. To complete the addition of products to the OSC, you need to provide asset-level metadata. <br>
There are three requirements for this:

1. **Storage**. Your research data and workflows/code must be hosted on remote, persistent storage that allows discovery. Examples include:

* Repository provided by ESA >>> see `2.1 Product_files_PRR.ipynb notebook` 
* S3-compatible object storage - permanent and public
* Zenodo, CEDA, Dataverse, or other persistent archives used by the academic community

2. **Format**. We encourage you to use [cloud-optimised data format](https://esa-earthcode.github.io/documentation/Community%20and%20Best%20Practices/Data%20and%20Workflow%20Best%20Practices/Data/), since it makes storage and access to the products much easier.

3.  **File-level metadata.** To add your data to the Open Science Catalog, you have to generate a STAC items that describes your files, code (if applicable) and documentation (if applicable).
- If you already cover requirement 1. and 2, you should now start to **Generate STAC Items for your dataset** >>> `guide/2.2.Product_files_self_hosted.ipynb`
<br>


ℹ️ **To request permanent storage of data on ESA cloud storage to preserve the data in long-term**, navigate to >>> `guide/2.1.Product_files_PRR.ipynb.`


> We can support you through this all stages of this process, just [**contact us**](mailto:earth-code.esa.int) or post in the [FORUM](https://discourse-earthcode.eox.at/c/technical-support/8)!